# 📖 Recipe Book Page Distribution

You are given the table with titles of recipes from a cookbook and their page numbers. You are asked to represent how the recipes will be distributed in the book.

Produce a table consisting of three columns: left_page_number, left_title and right_title. The k-th row (counting from 0), should contain the number and the title of the page with the number 2×k in the first and second columns respectively, and the title of the page with the number 2×k+1 in the third column.

Each page contains at most 1 recipe. If the page does not contain a recipe, the appropriate cell should remain empty (NULL value). Page 0 (the internal side of the front cover) is guaranteed to be empty.

🌀 Trust me, this one will surely challenge you...! You'll learn self join, subquery. Give it a try and share the output! 👇

In [0]:
%skip
%sql
CREATE TABLE ska_catalog.bronze.cookbook_titles (page_number INT PRIMARY KEY,title VARCHAR(255));

INSERT INTO  ska_catalog.bronze.cookbook_titles (page_number, title) VALUES (1, 'Scrambled eggs'), (2, 'Fondue'), (3, 'Sandwich'), (4, 'Tomato soup'), (6, 'Liver'), (11, 'Fried duck'), (12, 'Boiled duck'), (15, 'Baked chicken');

In [0]:
%sql
SELECT * FROM ska_catalog.bronze.cookbook_titles

In [0]:
%sql
SELECT page_number, title FROM ska_catalog.bronze.cookbook_titles
WHERE page_number % 2 = 1

In [0]:
%sql
SELECT page_number, title FROM ska_catalog.bronze.cookbook_titles
WHERE page_number % 2 = 0

In [0]:
%sql
SELECT
  left_pages.page_number , left_pages.title AS `left_title`, right_pages.title AS `right_title`
FROM
  (
    SELECT page_number, title FROM ska_catalog.bronze.cookbook_titles
    WHERE page_number % 2 = 0
  ) AS left_pages
LEFT JOIN
  (
    SELECT page_number, title FROM ska_catalog.bronze.cookbook_titles
    WHERE page_number % 2 = 1
  ) AS right_pages
ON left_pages.page_number + 1 = right_pages.page_number
ORDER BY left_pages.page_number

In [0]:
import pandas as pd

df_cookbook_title = spark.table('ska_catalog.bronze.cookbook_titles').toPandas()

# Separate left and right pages
left_pages = df_cookbook_title[df_cookbook_title['page_number'] % 2 == 0][['page_number', 'title']].rename(columns={'title': 'left_title'})
right_pages = df_cookbook_title[df_cookbook_title['page_number'] % 2 == 1][['page_number', 'title']].rename(columns={'page_number': 'right_page_number', 'title': 'right_title'})

# Adjust right_page_number to match left page_number
right_pages['left_page_number'] = right_pages['right_page_number'] - 1

# Merge left and right pages
result = pd.merge(left_pages, right_pages, left_on='page_number', right_on='left_page_number', how='left')

# Select and sort columns
result = result[['page_number', 'left_title', 'right_title']].sort_values('page_number')

display(result)